# Urdu voice clips for urdu_reels.py (Urdu Tortoise-TTS)
Uses the Urdu fine-tuned Tortoise model from **github.com/ahmedHanzala/urdu-tts** (weights: huggingface.co/zohann/urdu-tts).

1. **Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.
2. Upload your `videos.json` (the same file you give `urdu_reels.py`).
3. Optional: upload 3–5 short WAV samples (6–10 s each, clean, no music) of the voice you want cloned. Use **your own voice or one you have permission to use**.
4. Download `clips.zip`, unzip it next to `urdu_reels.py`, set `VOICE_ENGINE = "clips"` and render as usual.

Tortoise is slow: about 20–60 s per line on a T4 with the `fast` preset, longer with `standard`.


In [ ]:
#@title 1. Install (takes a few minutes)
!nvidia-smi -L
%cd /content
!git lfs install
!git clone https://huggingface.co/zohann/urdu-tts
!git clone https://github.com/152334H/tortoise-tts-fast
%cd /content/tortoise-tts-fast
!pip3 install -q -r requirements.txt --no-deps
!pip3 install -q -e .
!pip3 install -q git+https://github.com/152334H/BigVGAN.git
import glob
AR = sorted(glob.glob('/content/urdu-tts/**/*.pth', recursive=True))
print('checkpoints found:', AR)
AR = next((p for p in AR if 'urdu-multispeaker' in p), AR[0] if AR else None)
print('using:', AR)


In [ ]:
#@title 2. Settings + upload videos.json
PRESET = "fast"        #@param ["ultra_fast", "fast", "standard", "high_quality"]
CANDIDATES = 2         #@param {type:"integer"}  # takes per line: line_01.wav plus line_01_alt1.wav ...
VOICE = "daniel"       #@param {type:"string"}   # a built-in Tortoise voice, or "myvoice" after step 3
from google.colab import files
import json, os
os.chdir('/content')
up = files.upload()                      # choose videos.json
spec = json.load(open(next(iter(up)), encoding='utf-8'))
for v in spec['videos']:
    print(v['name'], '-', len(v['lines']), 'lines')


In [ ]:
#@title 3. (Optional) upload voice samples to clone -> VOICE = "myvoice"
import os, shutil
from google.colab import files
vdir = '/content/tortoise-tts-fast/tortoise/voices/myvoice'
os.makedirs(vdir, exist_ok=True)
for name, data in files.upload().items():
    open(os.path.join(vdir, name), 'wb').write(data)
VOICE = "myvoice"
print(os.listdir(vdir))


In [ ]:
#@title 4. Generate one clip per line
import os, re, torch, torchaudio
from IPython.display import Audio, display
os.chdir('/content/tortoise-tts-fast')
from tortoise.api import TextToSpeech
from tortoise.utils.audio import load_voice

tts = TextToSpeech(ar_checkpoint=AR)
voice_samples, latents = load_voice(VOICE)
OUT = '/content/clips'
for v in spec['videos']:
    d = os.path.join(OUT, v['name']); os.makedirs(d, exist_ok=True)
    for i, ln in enumerate(v['lines']):
        if not ln.get('speak', True):
            continue
        if ln.get('font') == 'arabic' and 'say_ur' not in ln:
            continue                       # never TTS-recite Quranic Arabic; narrate the Urdu translation
        text = ln.get('say_ur') or ln['text']
        text = re.sub(r'[\u064B-\u0652\u0670]', '', text)   # drop harakat
        print(f"{v['name']} line {i+1}: {text}")
        gen = tts.tts_with_preset(text, voice_samples=voice_samples, conditioning_latents=latents,
                                  preset=PRESET, k=CANDIDATES)
        gens = gen if isinstance(gen, (list, tuple)) else [gen]
        for c, g in enumerate(gens):
            p = os.path.join(d, f"line_{i+1:02d}{'' if c == 0 else f'_alt{c}'}.wav")
            torchaudio.save(p, g.squeeze(0).cpu(), 24000)
            display(Audio(p))


Listen to each take above. To use an alternative take, rename it: e.g. delete `line_03.wav` and rename `line_03_alt1.wav` to `line_03.wav` (Files panel on the left, or the next cell).

In [ ]:
#@title 5. (Optional) swap in an alternative take
VIDEO = ""   #@param {type:"string"}
LINE = 3     #@param {type:"integer"}
ALT = 1      #@param {type:"integer"}
import os
d = os.path.join('/content/clips', VIDEO)
os.replace(os.path.join(d, f'line_{LINE:02d}_alt{ALT}.wav'), os.path.join(d, f'line_{LINE:02d}.wav'))
print('done')


In [ ]:
#@title 6. Download clips.zip  (unzip next to urdu_reels.py so you get clips/<video name>/line_01.wav ...)
import shutil
from google.colab import files
shutil.make_archive('/content/clips', 'zip', '/content', 'clips')
files.download('/content/clips.zip')
